# Двумерная динамика поляритонного конденсата

Главный исследовательский ноутбук по ТЗ v3. Длительные ячейки запускаются только при явном изменении флага `DO_*`. Все физические вычисления находятся в модулях; рисунки читают сохранённые массивы.


## 1. Конфигурация и единицы


In [ ]:
from storage import load_config, check_config, config_hash
from model import parameters, make_grid, make_pump

config = load_config("configs/gaussian.yaml")
check_config(config)
p = parameters(config)
grid = make_grid(config)
pump = make_pump(config, grid)
print("config_sha256:", config_hash(config))
print("K [meV um²]:", p["K"], "Pth0D [um⁻² ps⁻¹]:", p["Pth0D"], "eta [ps]:", p["eta"])
print("grid:", grid.shape, "dx/dy [um]:", grid.dx, grid.dy, "Ptotal [ps⁻¹]:", pump.sum()*grid.da)


## 2. Быстрые численные проверки

Показываем измеренные невязки; автоматические тесты можно запустить отдельно.


In [ ]:
from numerics import run_validation_suite
from storage import load_config
report = run_validation_suite(load_config("configs/validation.yaml"))
for key, value in report.items():
    print(f"{key}: {value}")
assert report["PASS"]


In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "pytest", "-q"], check=False)
print("pytest exit code:", result.returncode)


## 3. Замер вычислительной машины

Прогноз времени является оценкой. Полные 2D-прогоны не запускаются этой ячейкой.


In [ ]:
DO_BENCHMARK = False
if DO_BENCHMARK:
    from numerics import benchmark
    from storage import load_config
    for item in benchmark(load_config("configs/gaussian.yaml"), steps=300, sizes=(256, 512, 1024)):
        print(item)


## 4. Однородная модель и Стюарт–Ландау

Сначала сохраняются числовые массивы, затем можно строить F1–F2 без 2D-расчёта.


In [ ]:
DO_0D = False
if DO_0D:
    from plotting import build_0d_data, make_figures
    from storage import load_config
    from pathlib import Path
    c = load_config("configs/gaussian.yaml")
    data_dir = build_0d_data(c, Path("results") / "analysis_data")
    print(make_figures(c, data_dir=data_dir, output_dir=Path("results") / "report_figures"))


## 5. Гауссова накачка

Результат и контрольные точки сохраняются в `results/<run_id>/`. Лимит 20 минут приводит к `RUN_INCOMPLETE`, который можно продолжить с той же конфигурацией.


In [ ]:
DO_GAUSSIAN = False
if DO_GAUSSIAN:
    from numerics import simulate_2d
    from storage import load_config
    gaussian_config = load_config("configs/gaussian.yaml")
    gaussian_config["pump"]["P0"] = 15.0  # пример выше 2D-порога
    gaussian_config["space"]["Nx"] = gaussian_config["space"]["Ny"] = 256
    gaussian_run = simulate_2d(gaussian_config)
    print(gaussian_run["path"], gaussian_run["summary"]["run_status"])


### 5a. Численный поглощающий слой (только при проверке границ)

Это отдельный численный сценарий. Его результат сравнивается с периодической областью большего размера, а потери слоя входят в баланс.


In [ ]:
DO_ABSORBER = False
if DO_ABSORBER:
    from numerics import simulate_2d
    from storage import load_config
    absorbing_config = load_config("configs/gaussian.yaml")
    absorbing_config["pump"]["P0"] = 15.0
    absorbing_config["space"]["Nx"] = absorbing_config["space"]["Ny"] = 256
    absorbing_config["space"].update(absorbing_layer=True, absorber_width_um=25.0, absorber_gamma_max_per_ps=2.0)
    absorbing_run = simulate_2d(absorbing_config)
    print(absorbing_run["path"], absorbing_run["summary"]["run_status"])


## 6. Кольцевая накачка


In [ ]:
DO_RING = False
if DO_RING:
    from numerics import simulate_2d
    from storage import load_config
    ring_config = load_config("configs/ring.yaml")
    ring_config["space"]["Nx"] = ring_config["space"]["Ny"] = 256
    ring_run = simulate_2d(ring_config)
    print(ring_run["path"], ring_run["summary"]["run_status"])


## 7. Линейные 2D-пороги

Здесь резервуар заранее установлен в `P/gammaR` и заморожен. Это отдельный протокол от физического включения накачки. `PRELIMINARY` требует проверок сетки и области.


In [ ]:
DO_THRESHOLDS = False
if DO_THRESHOLDS:
    from numerics import find_threshold_2d
    from storage import load_config
    gaussian_config = load_config("configs/gaussian.yaml")
    ring_config = load_config("configs/ring.yaml")
    for c in (gaussian_config, ring_config):
        c["space"]["Nx"] = c["space"]["Ny"] = 128
    gaussian_threshold = find_threshold_2d(gaussian_config)
    ring_threshold = find_threshold_2d(ring_config)
    print(gaussian_threshold["status"], ring_threshold["status"])


## 7a. Проверка порога на сетке и области; нелинейное подтверждение

Каждый блок запускается отдельно по каталогам сохранённых порогов. Нелинейное подтверждение делает два физических прогона для каждой геометрии.


In [ ]:
DO_THRESHOLD_VALIDATION = False
DO_NONLINEAR_CONFIRMATION = False
THRESHOLD_RUN_DIRS = []  # каталоги линейного порога для пятна и кольца
if DO_THRESHOLD_VALIDATION or DO_NONLINEAR_CONFIRMATION:
    from numerics import validate_threshold_2d, confirm_threshold_nonlinear
    from storage import load_run
    for directory in THRESHOLD_RUN_DIRS:
        saved = load_run(directory)
        threshold = dict(saved["summary"]["threshold"])
        threshold["run_dir"] = str(saved["path"])
        if DO_THRESHOLD_VALIDATION:
            print(saved["path"], validate_threshold_2d(saved["config"], threshold))
        if DO_NONLINEAR_CONFIRMATION:
            import copy
            physical_config = copy.deepcopy(saved["config"])
            physical_config["space"]["Nx"] = physical_config["space"]["Ny"] = 256
            print(saved["path"], confirm_threshold_nonlinear(physical_config, threshold))


## 8. Сходимость по сетке, времени и области

Большая серия запускается отдельно от обычных прогонов. Неполные расчёты отмечаются `NOT_TESTED`.


In [ ]:
DO_CONVERGENCE = False
if DO_CONVERGENCE:
    from numerics import convergence_study
    from storage import load_config
    convergence = convergence_study(load_config("configs/gaussian.yaml"))
    print(convergence["status"], convergence["run_dir"])


## 9. Продолжение прогона

Укажите существующий каталог с `RUN_INCOMPLETE`. Менять разрешено только конечное время в конфигурации.


In [ ]:
DO_RESUME = False
RUN_DIR_TO_RESUME = ""
if DO_RESUME:
    from numerics import simulate_2d
    from storage import load_config
    resumed = simulate_2d(load_config("configs/gaussian.yaml"), resume_dir=RUN_DIR_TO_RESUME)
    print(resumed["summary"]["run_status"])


## 10. Рисунки только из сохранённых числовых данных

Укажите каталоги законченных прогонов и JSON результатов порога/сходимости. Отсутствующие панели не создаются.


In [ ]:
DO_FIGURES = False
GAUSSIAN_RUN_DIR = ""
RING_RUN_DIR = ""
GAUSSIAN_THRESHOLD_JSON = ""
RING_THRESHOLD_JSON = ""
CONVERGENCE_JSON = ""
if DO_FIGURES:
    from plotting import make_figures
    from storage import load_config
    from pathlib import Path
    files = make_figures(load_config("configs/gaussian.yaml"),
        data_dir=Path("results") / "analysis_data",
        output_dir=Path("results") / "report_figures",
        gaussian_run=GAUSSIAN_RUN_DIR or None, ring_run=RING_RUN_DIR or None,
        gaussian_threshold=GAUSSIAN_THRESHOLD_JSON or None,
        ring_threshold=RING_THRESHOLD_JSON or None,
        convergence=CONVERGENCE_JSON or None)
    for file in files:
        print(file)
